# DataCo Supply Chain Analytics: SQL Business Analysis

This notebook uses SQLite to query the cleaned DataCo Supply Chain dataset. The goal is to recreate key business KPI summaries using SQL, including delivery performance, product profitability, and order status analysis.

## 1. Project Context

This notebook is the SQL-focused component of my DataCo Supply Chain Analytics portfolio project. The purpose of this notebook is to use SQLite to recreate key business KPI summaries from the main Python analysis.

The SQL analysis focuses on three major areas:

1. Delivery performance and customer experience risk
2. Product sales, profit, and margin performance
3. Operations and fulfillment outcomes

The cleaned dataset is loaded into a SQLite database table named `dataco_orders`, which allows the business questions to be answered using SQL queries.

In [1]:
import pandas as pd
import sqlite3
from google.colab import files

In [2]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_sql = pd.read_csv(filename)
df_sql.head()

Saving DataCo_SupplyChain_Dataset_Cleaned.csv to DataCo_SupplyChain_Dataset_Cleaned.csv


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_month,order_month_name,order_day_of_week,order_year_month,shipping_delay_days,is_late_delivery,delivery_status_clean,profit_margin,profit_margin_pct,cx_delivery_risk
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,1,January,Wednesday,2018-01,-1,0,On Time,0.278413,27.841342,Low Risk
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,1,January,Saturday,2018-01,1,1,Late,-0.760000,-75.999999,Moderate Risk
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,1,January,Saturday,2018-01,0,0,On Time,-0.756003,-75.600305,Low Risk
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,1,January,Saturday,2018-01,-1,0,On Time,0.069748,6.974829,Low Risk
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,1,January,Saturday,2018-01,-2,0,On Time,0.409489,40.948896,Low Risk


## 2. Create SQLite Database

In this section, I create a SQLite database and load the cleaned DataCo dataset into a table named `dataco_orders`. This allows the project’s business questions to be answered using SQL queries.

In [3]:
# Create SQLite database connection

conn = sqlite3.connect("dataco_supply_chain.db")

In [4]:
# Load cleaned dataset into SQLite table

df_sql.to_sql(
    "dataco_orders",
    conn,
    if_exists="replace",
    index=False
)

180519

In [5]:
# Confirming the table was created properly

query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, conn)

,name
0,dataco_orders


In [6]:
# Confirming row count

query = """
SELECT COUNT(*) AS total_rows
FROM dataco_orders;
"""

pd.read_sql_query(query, conn)

,total_rows
0,180519


In [7]:
# Preview first 5 records from the SQL table

query = """
SELECT *
FROM dataco_orders
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_month,order_month_name,order_day_of_week,order_year_month,shipping_delay_days,is_late_delivery,delivery_status_clean,profit_margin,profit_margin_pct,cx_delivery_risk
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,1,January,Wednesday,2018-01,-1,0,On Time,0.278413,27.841342,Low Risk
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,1,January,Saturday,2018-01,1,1,Late,-0.760000,-75.999999,Moderate Risk
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,1,January,Saturday,2018-01,0,0,On Time,-0.756003,-75.600305,Low Risk
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,1,January,Saturday,2018-01,-1,0,On Time,0.069748,6.974829,Low Risk
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,1,January,Saturday,2018-01,-2,0,On Time,0.409489,40.948896,Low Risk


## 3. Executive KPI Summary

This query calculates the overall business KPIs for the cleaned DataCo dataset, including total orders, late delivery rate, total sales, total profit, average shipping days, and average shipping delay.

In [8]:
query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(is_late_delivery) AS late_orders,
    COUNT(*) - SUM(is_late_delivery) AS on_time_orders,
    ROUND(SUM(is_late_delivery) * 100.0 / COUNT(*), 2) AS late_delivery_rate_pct,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(days_for_shipping_real), 2) AS avg_actual_shipping_days,
    ROUND(AVG(days_for_shipment_scheduled), 2) AS avg_scheduled_shipping_days,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay
FROM dataco_orders;
"""

pd.read_sql_query(query, conn)

,total_orders,late_orders,on_time_orders,late_delivery_rate_pct,total_sales,total_profit,avg_actual_shipping_days,avg_scheduled_shipping_days,avg_shipping_delay
0,180519,98977,81542,54.83,36784735.01,3966902.97,3.5,2.93,0.57


### Executive KPI Summary Interpretation

The SQL query confirms that the cleaned DataCo dataset contains 180,519 total orders. Of those orders, 98,977 were flagged as late, producing an overall late delivery rate of 54.83%.

The dataset includes approximately \$36.8 million in total sales and approximately \$4.0 million in total profit. On average, orders took 3.50 days to ship compared with 2.93 scheduled shipping days, creating an average shipping delay of 0.57 days.

This confirms that late delivery is a major customer experience and operations issue in the dataset.

## 4. Delivery Performance SQL Analysis

This section uses SQL to compare late delivery performance across shipping modes, customer segments, markets, and order regions. The goal is to identify whether delivery problems are concentrated in specific fulfillment methods, customer groups, or geographic areas.

### 4.1 Late Delivery Rate by Shipping Mode

This query compares late delivery rate, average shipping delay, total sales, and total profit by shipping mode.

In [9]:
query = """
SELECT
    shipping_mode,
    COUNT(*) AS total_orders,
    SUM(is_late_delivery) AS late_orders,
    COUNT(*) - SUM(is_late_delivery) AS on_time_orders,
    ROUND(SUM(is_late_delivery) * 100.0 / COUNT(*), 2) AS late_delivery_rate_pct,
    ROUND(AVG(days_for_shipping_real), 2) AS avg_actual_shipping_days,
    ROUND(AVG(days_for_shipment_scheduled), 2) AS avg_scheduled_shipping_days,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit
FROM dataco_orders
GROUP BY shipping_mode
ORDER BY late_delivery_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,shipping_mode,total_orders,late_orders,on_time_orders,late_delivery_rate_pct,avg_actual_shipping_days,avg_scheduled_shipping_days,avg_shipping_delay,total_sales,total_profit
0,First Class,27814,26513,1301,95.32,2.00,1.0,1.00,5674369.76,643121.92
1,Second Class,35216,26987,8229,76.63,3.99,2.0,1.99,7145444.82,750308.17
2,Same Day,9737,4454,5283,45.74,0.48,0.0,0.48,1942528.56,203018.43
3,Standard Class,107752,41023,66729,38.07,4.00,4.0,-0.00,22022391.88,2370454.45


### Interpretation

The SQL output shows that First Class and Second Class shipping have the highest late delivery rates. First Class orders were late 95.32% of the time, while Second Class orders were late 76.63% of the time.

This suggests that faster shipping promises may be difficult to meet consistently in this dataset. Standard Class had the lowest late delivery rate at 38.07%, despite having the largest order volume.

From a customer experience and operations perspective, this indicates that shipping mode is one of the strongest indicators of delivery risk. The business may need to review whether First Class and Second Class delivery timelines are realistic or whether fulfillment processes need improvement for expedited shipping.

### 4.2 Late Delivery Rate by Customer Segment

This query compares late delivery rates across customer segments to determine whether delivery problems are concentrated among specific customer groups.

In [10]:
query = """
SELECT
    customer_segment,
    COUNT(*) AS total_orders,
    SUM(is_late_delivery) AS late_orders,
    COUNT(*) - SUM(is_late_delivery) AS on_time_orders,
    ROUND(SUM(is_late_delivery) * 100.0 / COUNT(*), 2) AS late_delivery_rate_pct,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit
FROM dataco_orders
GROUP BY customer_segment
ORDER BY late_delivery_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,customer_segment,total_orders,late_orders,on_time_orders,late_delivery_rate_pct,avg_shipping_delay,total_sales,total_profit
0,Home Office,32226,17747,14479,55.07,0.58,6520538.02,690840.34
1,Consumer,93504,51248,42256,54.81,0.56,19095790.16,2073487.67
2,Corporate,54789,29982,24807,54.72,0.56,11168406.84,1202574.96


### Interpretation

The SQL output shows that late delivery rates are very similar across all customer segments. Home Office customers had the highest late delivery rate at 55.07%, followed by Consumer customers at 54.81% and Corporate customers at 54.72%.

Because the differences between segments are small, this suggests that late delivery is not concentrated within one specific customer group. Instead, the issue appears to be broader and more operational in nature.

This supports the earlier finding that shipping mode is a stronger indicator of delivery risk than customer segment.

### 4.3 Late Delivery Rate by Market

This query compares late delivery rates across markets to determine whether delivery issues are concentrated in specific geographic areas.

In [11]:
query = """
SELECT
    market,
    COUNT(*) AS total_orders,
    SUM(is_late_delivery) AS late_orders,
    COUNT(*) - SUM(is_late_delivery) AS on_time_orders,
    ROUND(SUM(is_late_delivery) * 100.0 / COUNT(*), 2) AS late_delivery_rate_pct,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit
FROM dataco_orders
GROUP BY market
ORDER BY late_delivery_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,market,total_orders,late_orders,on_time_orders,late_delivery_rate_pct,avg_shipping_delay,total_sales,total_profit
0,Europe,50252,27743,22509,55.21,0.57,10872396.80,1169442.96
1,Pacific Asia,41260,22712,18548,55.05,0.57,8273743.74,857753.44
2,USCA,25799,14138,11661,54.80,0.57,5066528.71,564313.78
3,Africa,11614,6340,5274,54.59,0.56,2294452.93,252071.18
4,LATAM,51594,28044,23550,54.36,0.56,10277612.84,1123321.61


### Interpretation

The SQL output shows that late delivery rates are very similar across markets. Europe had the highest late delivery rate at 55.21%, while LATAM had the lowest at 54.36%.

Because the difference between the highest and lowest market is less than one percentage point, delivery issues do not appear to be strongly concentrated in one broad geographic market.

This supports the earlier finding that late delivery is likely more related to fulfillment expectations, shipping mode, or operational process issues than broad market location.

### 4.4 Late Delivery Rate by Order Region

This query compares late delivery rates by order region. Because there are many regions, the output is limited to the top 10 regions with the highest late delivery rates.

In [12]:
query = """
SELECT
    order_region,
    COUNT(*) AS total_orders,
    SUM(is_late_delivery) AS late_orders,
    COUNT(*) - SUM(is_late_delivery) AS on_time_orders,
    ROUND(SUM(is_late_delivery) * 100.0 / COUNT(*), 2) AS late_delivery_rate_pct,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit
FROM dataco_orders
GROUP BY order_region
ORDER BY late_delivery_rate_pct DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,order_region,total_orders,late_orders,on_time_orders,late_delivery_rate_pct,avg_shipping_delay,total_sales,total_profit
0,Central Africa,1677,972,705,57.96,0.64,327263.02,33447.27
1,South Asia,7731,4350,3381,56.27,0.60,1553680.92,165703.90
2,East Africa,1852,1036,816,55.94,0.57,376234.90,43167.73
3,Western Europe,27109,15140,11969,55.85,0.60,5894380.77,625446.08
4,South of USA,4045,2256,1789,55.77,0.58,785783.95,88114.88
5,Eastern Europe,3920,2182,1738,55.66,0.58,774266.57,79717.05
6,East of USA,6915,3849,3066,55.66,0.58,1371111.99,156263.30
7,Southeast Asia,9539,5297,4242,55.53,0.56,1932495.57,211342.82
8,Central Asia,553,306,247,55.33,0.65,109839.93,13045.28
9,West Asia,6009,3322,2687,55.28,0.57,1174671.78,118815.41


### Interpretation

The SQL output shows that Central Africa had the highest late delivery rate among order regions at 57.96%, followed by South Asia at 56.27% and East Africa at 55.94%.

Although these regions rank highest, the differences are still relatively small compared with the overall late delivery rate of 54.83%. This suggests that some regions may experience slightly higher delivery risk, but region alone does not appear to explain the larger late delivery problem.

This supports the broader finding that shipping mode and fulfillment expectations are stronger indicators of delivery risk than geography.

## 5. Product and Profitability SQL Analysis

This section uses SQL to compare product categories by sales, profit, profit margin, discount rate, and late delivery rate. The goal is to identify which product categories drive business performance and which categories may need closer review.

### 5.1 Product Category Sales and Profitability

This query summarizes total orders, total sales, total profit, average profit margin, average discount rate, and late delivery rate by product category.

In [13]:
query = """
SELECT
    category_name,
    COUNT(*) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(profit_margin_pct), 2) AS avg_profit_margin_pct,
    ROUND(AVG(order_item_discount_rate) * 100, 2) AS avg_discount_rate_pct,
    ROUND(AVG(is_late_delivery) * 100, 2) AS late_delivery_rate_pct
FROM dataco_orders
GROUP BY category_name
ORDER BY total_sales DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category_name,total_orders,total_sales,total_profit,avg_profit_margin_pct,avg_discount_rate_pct,late_delivery_rate_pct
0,Fishing,17325,6929653.69,756220.77,10.91,10.17,54.93
1,Cleats,24551,4431942.78,494636.92,11.19,10.16,54.97
2,Camping & Hiking,13729,4118425.57,427455.57,10.38,10.17,54.53
3,Cardio Equipment,12487,3694843.20,383011.10,10.62,10.17,54.50
4,Women's Apparel,21035,3147800.00,350421.03,11.00,10.17,54.56
5,Water Sports,15540,3113844.68,325146.96,10.45,10.17,54.81
6,Men's Footwear,22246,2891757.66,311902.82,10.79,10.17,54.49
7,Indoor/Outdoor Games,19298,2888993.91,318451.43,11.13,10.15,54.75
8,Shop By Sport,10984,1309522.04,129813.96,10.09,10.14,55.15
9,Computers,442,663000.00,69656.81,10.51,10.22,50.68


### Interpretation

The SQL output shows that Fishing is the highest-sales product category, generating approximately \$6.9 million in sales and \$756,221 in profit. Cleats and Camping & Hiking are also major revenue drivers, with approximately \$4.4 million and \$4.1 million in sales respectively.

The top-selling product categories generally have similar average profit margins, mostly around 10% to 11%, and similar average discount rates around 10%. This suggests that differences in total sales and profit are mostly driven by order volume and category demand rather than major differences in margin or discounting.

Late delivery rates are also similar across these high-sales categories, generally around 54% to 55%, which means that delivery risk affects several of the business’s most important revenue categories.

### 5.2 Highest Average Profit Margin by Product Category

This query identifies the product categories with the highest average profit margins. This helps separate high-revenue categories from high-margin categories.

In [14]:
query = """
SELECT
    category_name,
    COUNT(*) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(profit_margin_pct), 2) AS avg_profit_margin_pct,
    ROUND(AVG(order_item_discount_rate) * 100, 2) AS avg_discount_rate_pct,
    ROUND(AVG(is_late_delivery) * 100, 2) AS late_delivery_rate_pct
FROM dataco_orders
GROUP BY category_name
ORDER BY avg_profit_margin_pct DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category_name,total_orders,total_sales,total_profit,avg_profit_margin_pct,avg_discount_rate_pct,late_delivery_rate_pct
0,Golf Bags & Carts,61,10369.39,1810.07,17.46,9.34,68.85
1,Toys,529,6104.66,900.71,14.75,10.17,55.01
2,Women's Clothing,650,140283.00,19102.85,13.62,10.20,56.46
3,Fitness Accessories,309,35601.44,5258.39,13.56,10.26,56.96
4,Men's Golf Clubs,283,47035.80,5517.99,13.17,9.91,47.70
5,Soccer,138,26477.05,3901.95,13.10,9.61,54.35
6,Garden,484,257768.73,33443.01,12.97,10.12,55.79
7,Women's Golf Clubs,181,44545.97,5028.64,12.93,10.17,53.59
8,Music,434,113122.10,14436.32,12.76,10.14,57.14
9,CDs,271,3059.59,383.85,12.55,10.15,52.03


### Interpretation

The SQL output shows that the highest-margin categories are not the same as the highest-sales categories. Golf Bags & Carts had the highest average profit margin at 17.46%, but it only had 61 orders, meaning it is a high-margin but low-volume category.

Several other high-margin categories, such as Toys, Women's Clothing, Fitness Accessories, and Garden, also generated much lower total sales than the top revenue categories from the previous query.

This distinction is important because high margin does not always mean high overall business impact. A category can be profitable on a percentage basis while still contributing a relatively small amount of total sales or total profit because of low order volume.

Golf Bags & Carts may deserve closer review because it combines the highest average profit margin with a very high late delivery rate of 68.85%.

## 6. Operations and Fulfillment SQL Analysis

This section uses SQL to analyze order status, sales, profit, and fulfillment risk. The goal is to understand how different order outcomes, such as completed, pending, canceled, or suspected fraud orders, relate to operational performance and business value.

### 7.1 Order Status Summary

This query summarizes order count, order share, total sales, total profit, late delivery rate, and average shipping delay by order status.

In [15]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM dataco_orders), 2) AS order_share_pct,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(is_late_delivery) * 100, 2) AS late_delivery_rate_pct,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay
FROM dataco_orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

pd.read_sql_query(query, conn)

,order_status,total_orders,order_share_pct,total_sales,total_profit,late_delivery_rate_pct,avg_shipping_delay
0,COMPLETE,59491,32.96,12095314.95,1321735.75,57.49,0.56
1,PENDING_PAYMENT,39832,22.07,8106697.56,843810.24,57.55,0.59
2,PROCESSING,21902,12.13,4504063.75,494825.87,57.09,0.56
3,PENDING,20227,11.20,4120532.87,435725.85,57.90,0.57
4,CLOSED,19616,10.87,4022624.17,457981.09,56.63,0.55
5,ON_HOLD,9804,5.43,1981542.71,208913.04,55.59,0.53
6,SUSPECTED_FRAUD,4062,2.25,825934.96,85136.71,0.00,0.59
7,CANCELED,3692,2.05,744370.40,75345.63,0.00,0.56
8,PAYMENT_REVIEW,1893,1.05,383653.66,43428.79,57.16,0.60


### Interpretation

The SQL output shows that COMPLETE orders make up the largest share of the dataset at 32.96% of total orders. PENDING_PAYMENT is the second-largest order status at 22.07%, representing a major share of both order volume and revenue.

PENDING_PAYMENT orders generated approximately \$8.1 million in sales and \$843,810 in profit, which suggests that unresolved payment workflows may represent a meaningful operational and financial area to monitor.

Late delivery rates are similar across the actively fulfilled order statuses, generally around 56% to 58%. CANCELED and SUSPECTED_FRAUD orders show a 0.00% late delivery rate, which is expected because these orders likely did not move through the normal fulfillment and delivery process.

From an operations perspective, order status does not appear to be the main driver of late delivery risk. However, high-volume and high-value statuses such as COMPLETE and PENDING_PAYMENT should be prioritized because they represent the largest operational impact.

## 7. SQL Analysis Summary

The SQL analysis confirms the main findings from the Python business analysis notebook. The overall late delivery rate is 54.83%, showing that delivery performance is a major customer experience and operations issue in the dataset.

Shipping mode appears to be one of the strongest indicators of delivery risk. First Class and Second Class shipping had much higher late delivery rates than Standard Class, suggesting that faster shipping promises may be difficult to meet consistently.

Customer segment and broad market geography showed very similar late delivery rates, which suggests that late delivery is not isolated to one customer group or market.

The product analysis showed that the highest-sales categories are not always the highest-margin categories. Fishing generated the most sales and profit overall, while Golf Bags & Carts had the highest average profit margin but low order volume and high delivery risk.

The operations analysis showed that COMPLETE and PENDING_PAYMENT orders represent the largest shares of order volume and revenue. This makes payment resolution and fulfillment monitoring important areas for operational improvement.

Overall, this SQL notebook demonstrates how the major KPI summaries from the DataCo Supply Chain Analytics project can be recreated using SQL.